## Entorno: Data, docker y spark

Se descarga la data y se configura el entorno con docker y spark

#### Installs

Instala Jupyter (`notebook`), la dependencia necesaria para poder abrir y correr este notebook

In [1]:
%pip install -q notebook


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


#### Imports

Importa las librerías estándar (json, subprocess, urllib, etc.) que usan las demás celdas del notebook

In [10]:
import json, os, subprocess, sys, time, urllib.request
from pathlib import Path

#### Utilidades

Define `sh()`, la función que ejecuta comandos de shell mostrando su salida en vivo, y calcula la ruta raíz del proyecto

In [ ]:
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, cwd=proyecto, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for linea in p.stdout:
        print(linea, end="")

    p.wait()

    if p.returncode != 0:
        raise RuntimeError(p.returncode)

proyecto = Path.cwd()
if not (proyecto / "docker-compose.yml").exists() and (proyecto.parent / "docker-compose.yml").exists():
    proyecto = proyecto.parent

### 1. Descargar data

Ejecuta `download.py` para descargar los archivos JSON de contrataciones que se van a analizar

In [12]:
sh("python3 download.py")

### 2. Verificación docker

Comprueba que el CLI de Docker y el plugin `compose` estén instalados en la máquina

In [13]:
sh("docker --version")
sh("docker compose version")

Docker version 29.7.2, build a7dcaa6
Docker Compose version v5.4.0


### 3. Iniciar cluster spark

Levanta el cluster de Spark con `docker compose up -d` y muestra el estado de los contenedores.

In [14]:
sh("docker compose up -d")
sh("docker compose ps")

 Image quay.io/jupyter/pyspark-notebook:spark-3.5.0 Pulling 
 Image quay.io/jupyter/pyspark-notebook:spark-3.5.0 Pulling 
 Image quay.io/jupyter/pyspark-notebook:spark-3.5.0 Pulling 
 3e5db86eb9ec Pulling fs layer 0B
 4c3a00622f0d Pulling fs layer 0B
 098ed84bfb8f Pulling fs layer 0B
 25123bbcc3c7 Pulling fs layer 0B
 ad16de47eb80 Pulling fs layer 0B
 2328ab3b01c9 Pulling fs layer 0B
 0d9313aabaee Pulling fs layer 0B
 4f4fb700ef54 Pulling fs layer 0B
 e97f6a9aae74 Pulling fs layer 0B
 59b2eac333fe Pulling fs layer 0B
 c7b5d65b59f3 Pulling fs layer 0B
 1d67b1c7da9a Pulling fs layer 0B
 fc3b4873b0ba Pulling fs layer 0B
 6edecb3c2d77 Pulling fs layer 0B
 af63d9e06a1b Pulling fs layer 0B
 be2b1b735650 Pulling fs layer 0B
 54c8a0f499e4 Pulling fs layer 0B
 664a18c5d4d4 Pulling fs layer 0B
 6311e155bc25 Pulling fs layer 0B
 98d30defa6ff Pulling fs layer 0B
 9fa1b0ffeef3 Pulling fs layer 0B
 573ebce36e48 Pulling fs layer 0B
 565a45cdf298 Pulling fs layer 0B
 bedbdfc2978c Pulling fs layer 0B
 

### 4. Monitorear cluster

Muestra el estado de los contenedores y el consumo de CPU/RAM del cluster de Spark

In [15]:
sh("docker compose ps")
sh("docker stats --no-stream")

NAME                      IMAGE                                          COMMAND                  SERVICE        CREATED          STATUS                      PORTS
dm-proj1-jupyter-1        quay.io/jupyter/pyspark-notebook:spark-3.5.0   "tini -g -- start.sh…"   jupyter        18 seconds ago   Up 17 seconds (healthy)     0.0.0.0:4040->4040/tcp, [::]:4040->4040/tcp, 0.0.0.0:8888->8888/tcp, [::]:8888->8888/tcp
dm-proj1-spark-master-1   quay.io/jupyter/pyspark-notebook:spark-3.5.0   "tini -g -- start.sh…"   spark-master   19 seconds ago   Up 17 seconds (unhealthy)   0.0.0.0:8080->8080/tcp, [::]:8080->8080/tcp
dm-proj1-spark-worker-1   quay.io/jupyter/pyspark-notebook:spark-3.5.0   "tini -g -- start.sh…"   spark-worker   18 seconds ago   Up 17 seconds (unhealthy)   0.0.0.0:8081->8081/tcp, [::]:8081->8081/tcp
CONTAINER ID   NAME                      CPU %     MEM USAGE / LIMIT     MEM %     NET I/O           BLOCK I/O     PIDS
2408d7af708f   dm-proj1-jupyter-1        13.03%    78.44MiB / 7.6

### 5. Apagar cluster

Detiene y borra los contenedores con `docker compose down`, pero la data y los resultados quedan en disco

In [ ]:
# sh("docker compose down")